# Interpretable Machine Learning Regression for Nuclear Applications with Kolmogorov-Arnold Networks (KAN)

**Paper:** Erdem, O., Panczyk, N., Radaideh, M.I. (2025). *Interpretable Machine Learning Regression for Nuclear Applications with Kolmogorov-Arnold Networks (KAN).* International Conference on Mathematics and Computational Methods Applied to Nuclear Science and Engineering (M&C 2025), Denver, CO.

**Carpeta origen:** `Ciencia, energía nuclear y química/MandC2025_KAN_Nuclear.pdf`

## Como se usan las KAN en este paper

Este paper compara KAN frente a redes feedforward tradicionales (FNN) en dos problemas de regresion de ingenieria nuclear, evaluando no solo precision sino tambien **interpretabilidad** (la arquitectura de la KAN se puede convertir en una ecuacion simbolica cerrada) y **explicabilidad post-hoc** (mediante SHAP). Usa directamente la libreria `PyKAN` (Liu et al. 2024), sin proponer una variante arquitectonica nueva.

**Arquitectura KAN (Sec. 2, Ec. 4-6).** Cada arista de una capa KAN lleva una funcion univariada entrenable, suma de una funcion base tipo residual y una spline B:

$$\phi(x) = w_b\, b(x) + w_s\,\mathrm{spline}(x), \qquad b(x)=\mathrm{silu}(x)=\frac{x}{1+e^{-x}}, \qquad \mathrm{spline}(x)=\sum_i c_i B_i(x)$$

y una KAN completa de $L$ capas compone estas matrices de funciones univariadas $\mathbf{\Phi}_l$ capa a capa: $\mathrm{KAN}(\mathbf{x}) = (\mathbf{\Phi}_{L-1}\circ\cdots\circ\mathbf{\Phi}_0)\,\mathbf{x}$.

**Metodologia de entrenamiento (Sec. 4.2).** Para cada problema, el paper entrena una KAN inicial con una grid de $G=3$ intervalos durante 100 epocas, la **poda** (elimina aristas con puntuacion $<0.03$ y nodos con puntuacion $<0.01$, los umbrales por defecto de `PyKAN`), continua entrenando la red podada 100 epocas mas, y despues **refina la grid** progresivamente ($G=5\to10\to25$, 100 epocas cada etapa), acumulando 500 epocas en total. Tras esto, cada arista superviviente se ajusta a una funcion de una biblioteca simbolica (`SYMBOLIC_LIB` de `PyKAN`: `sin`, `cos`, `tan`, `exp`, `log`, `tanh`, polinomios, etc.), y la red completa se convierte en una **formula cerrada**.

**Dos casos de estudio.** (1) **HTGR (micro-reactor HOLOS-Quad):** predecir el flujo neutronico normalizado en el primer cuadrante ($Q1$) a partir de los angulos de 8 tambores de control ($\theta_1,\ldots,\theta_8$), con arquitectura KAN $[8,4]$ (751 muestras, del benchmark `pyMAISE`). (2) **CHF (flujo critico de calor):** predecir el CHF a partir de 6 variables termohidraulicas (diametro $D$, longitud calentada $L$, presion $P$, flujo masico $G$, temperatura de entrada $T_{in}$, calidad de equilibrio $X_e$), con arquitectura KAN $[6,1,1]$, sobre el benchmark de la NRC/NEA (Groeneveld 2006). El paper publica explicitamente las ecuaciones simbolicas resultantes (Ec. 7 para HTGR, Ec. 8 para CHF), que usamos en este cuaderno como referencia directa de comparacion.

**Explicabilidad (Sec. 4.3).** Ademas de la conversion simbolica, el paper aplica **Kernel SHAP** sobre la KAN y **DeepLIFT** sobre la FNN para comparar que variables considera importantes cada modelo, encontrando que la KAN tiende a coincidir mas con la fisica esperada del problema (p. ej. los tambores 1 y 2, mas cercanos al cuadrante Q1, resultan mas importantes).

Este cuaderno reproduce fielmente la capa KAN con splines B (Ec. 4-6), el protocolo completo de entrenamiento-poda-refinamiento de grid (Sec. 4.2), la conversion a formula simbolica, y una version simplificada de la comparacion de importancia de variables (Sec. 4.3), para **ambos** casos de estudio (HTGR y CHF), usando los propios datasets publicos del benchmark `pyMAISE` cuando estan disponibles.

## Repositorio publico

El propio paper (M&C 2025) no incluye un enlace a un repositorio de GitHub especifico en el texto ni en las referencias. Sin embargo, la busqueda en GitHub localizo:

- **aims-umich/2025-panczyk-kan** &mdash; https://github.com/aims-umich/2025-panczyk-kan &mdash; repositorio oficial del mismo grupo de autores (Panczyk, Erdem, Radaideh) que reproduce el articulo extendido *"Opening the AI black-box: Symbolic regression with Kolmogorov-Arnold Networks for advanced energy applications"* (Energy and AI, 2025), version de revista de este mismo trabajo de M&C 2025, con los mismos dos casos de estudio (HTGR y CHF) entre sus 8 datasets. Su propio README declara explicitamente: *"Since the real CHF dataset used in this analysis is not public, we have provided synthetic versions of this dataset."*
- **aims-umich/pyMAISE** &mdash; https://github.com/aims-umich/pyMAISE &mdash; la libreria citada como referencia [6] del paper, que aloja los datasets originales (`microreactor.csv` para HTGR, 751 muestras; `chf_train_synth.csv`/`chf_test_synth.csv` para CHF) publicamente en Zenodo (DOI 10.5281/zenodo.20140559). Este cuaderno **descarga estos mismos ficheros directamente** al ejecutarse.
- **KindXiaoming/pykan** &mdash; https://github.com/KindXiaoming/pykan &mdash; la libreria `PyKAN` que el paper usa directamente para construir, entrenar, podar y convertir a simbolico sus modelos KAN (version 0.2.8, ya clonada localmente en `Kolmogorov-Arnold Networks/codigo/pykan`, tambien instalable via `pip install pykan`).

En este cuaderno **no usamos `pykan` como caja negra**: siguiendo la misma convencion pedagogica que el resto de esta coleccion (p. ej. `KAN Symbolic Regression.ipynb`, `MultKAN Conserved Quantities.ipynb`), reimplementamos directamente en PyTorch el mecanismo central que describe el paper (splines B por recursion de Cox-de Boor, poda de aristas/nodos con los umbrales exactos 0.03/0.01, extension de grid, e identificacion simbolica por biblioteca de funciones candidatas), para maxima transparencia sobre cada paso.

In [ ]:
%pip install -q torch numpy pandas matplotlib scipy sympy

In [ ]:
import io
import urllib.request

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import sympy as sp

torch.manual_seed(0)
np.random.seed(0)
device = torch.device('cpu')          # las KAN de este tamano no se benefician de GPU
torch.set_default_dtype(torch.float64)  # el paper y PyKAN usan float64 para las splines
print('Device:', device)

## 1. Capa KAN con splines B (Ec. 4-6) y mascara de aristas para la poda

Implementamos directamente en PyTorch el mecanismo central que describe la Seccion 2 del paper: cada arista de la red lleva una funcion univariada entrenable $\phi(x)=w_b\,\mathrm{silu}(x)+w_s\sum_i c_i B_i(x)$ (Ec. 4-6), donde $B_i$ son bases B-spline evaluadas mediante la recursion de Cox-de Boor. Anadimos ademas un **buffer `edge_mask`** por capa (no entrenable): al ponerse a cero para una arista concreta, esa arista deja de contribuir a la suma del nodo, lo que nos permite implementar la **poda de aristas** de la Seccion 4.2 sin tener que reconstruir la red. `curve2coef`/`refine_grid` implementan la **extension de grid** (ajuste de una spline mas fina por minimos cuadrados para reproducir la curva actual), que es como el paper refina $G:3\to5\to10\to25$ sin reentrenar desde cero. Esta logica sigue la misma convencion que `KAN Symbolic Regression.ipynb` y `MultKAN Conserved Quantities.ipynb` de esta coleccion, que a su vez siguen `kan/spline.py` del repositorio oficial `pykan`.

In [ ]:
def B_batch(x, grid, k=0):
    """Evalua x en las bases B-spline de orden k mediante la recursion de Cox-de Boor.
    x: (batch, in_dim). grid: (in_dim, n_grid_points). Devuelve (batch, in_dim, n_bases)."""
    x = x.unsqueeze(2)
    grid = grid.unsqueeze(0)
    if k == 0:
        value = ((x >= grid[:, :, :-1]) & (x < grid[:, :, 1:])).to(x.dtype)
    else:
        B_km1 = B_batch(x[:, :, 0], grid=grid[0], k=k - 1)
        left = (x - grid[:, :, :-(k + 1)]) / (grid[:, :, k:-1] - grid[:, :, :-(k + 1)]) * B_km1[:, :, :-1]
        right = (grid[:, :, k + 1:] - x) / (grid[:, :, k + 1:] - grid[:, :, 1:-k]) * B_km1[:, :, 1:]
        value = left + right
    return torch.nan_to_num(value)


def coef2curve(x_eval, grid, coef, k):
    """Convierte coeficientes de spline en la curva evaluada en x_eval (Ec. 6)."""
    b_splines = B_batch(x_eval, grid, k=k)
    return torch.einsum('bik,iok->bio', b_splines, coef)


def curve2coef(x_eval, y_eval, grid, k):
    """Ajusta coeficientes de spline por minimos cuadrados a partir de muestras (x,y):
    paso central de la extension de grid (Sec. 4.2)."""
    batch, in_dim = x_eval.shape
    out_dim = y_eval.shape[2]
    n_coef = grid.shape[1] - k - 1
    mat = B_batch(x_eval, grid, k=k)
    mat = mat.permute(1, 0, 2).unsqueeze(1).expand(in_dim, out_dim, batch, n_coef)
    y = y_eval.permute(1, 2, 0).unsqueeze(3)
    coef = torch.linalg.lstsq(mat, y).solution[:, :, :, 0]
    return coef


def extend_grid(grid, k_extend=0):
    """Extiende la grid k puntos a cada lado para que las splines de orden k esten bien definidas en los bordes."""
    h = (grid[:, [-1]] - grid[:, [0]]) / (grid.shape[1] - 1)
    for _ in range(k_extend):
        grid = torch.cat([grid[:, [0]] - h, grid], dim=1)
        grid = torch.cat([grid, grid[:, [-1]] + h], dim=1)
    return grid


class KANLayer(nn.Module):
    """Una capa KAN: activaciones aprendibles phi_{j,i}(x_i) = w_b*silu(x_i) + w_s*spline(x_i)
    por cada arista (Ec. 4-6), con una mascara de aristas (edge_mask) para la poda (Sec. 4.2)."""

    def __init__(self, in_dim, out_dim, grid_size=3, k=3, grid_range=(0, 1)):
        super().__init__()
        self.in_dim, self.out_dim, self.k = in_dim, out_dim, k
        grid = torch.linspace(grid_range[0], grid_range[1], grid_size + 1).unsqueeze(0).repeat(in_dim, 1)
        self.grid = extend_grid(grid, k_extend=k)                # no entrenable
        n_coef = self.grid.shape[1] - k - 1
        self.coef = nn.Parameter(torch.randn(in_dim, out_dim, n_coef) * 0.1)   # spline(x) ~ 0 al inicio
        self.scale_base = nn.Parameter(torch.empty(in_dim, out_dim).uniform_(-1, 1) / np.sqrt(in_dim))
        self.scale_spline = nn.Parameter(torch.ones(in_dim, out_dim))
        self.register_buffer('edge_mask', torch.ones(in_dim, out_dim))          # 1 = arista activa

    def forward(self, x):
        base = torch.nn.functional.silu(x)                        # b(x) = silu(x), Ec. 5
        spline = coef2curve(x, self.grid, self.coef, self.k)       # (batch, in_dim, out_dim)
        phi = self.scale_base.unsqueeze(0) * base.unsqueeze(2) + self.scale_spline.unsqueeze(0) * spline
        phi = phi * self.edge_mask.unsqueeze(0)                    # aristas podadas -> contribucion nula
        y = phi.sum(dim=1)                                          # suma en el nodo, Ec. 4
        return y, phi

    @torch.no_grad()
    def refine_grid(self, x_sample, new_grid_size):
        """Extension de grid: ajusta una grid mas fina que reproduce la curva actual por minimos cuadrados."""
        _, phi_old = self.forward(x_sample)
        lo = self.grid[0, self.k].item()
        hi = self.grid[0, -self.k - 1].item()
        new_grid = torch.linspace(lo, hi, new_grid_size + 1).unsqueeze(0).repeat(self.in_dim, 1)
        new_grid = extend_grid(new_grid, k_extend=self.k)
        base_old = torch.nn.functional.silu(x_sample)
        spline_target = phi_old - self.scale_base.unsqueeze(0) * base_old.unsqueeze(2)
        spline_target = spline_target * self.edge_mask.unsqueeze(0)
        new_coef = curve2coef(x_sample, spline_target, new_grid, self.k)
        self.grid = new_grid
        self.coef = nn.Parameter(new_coef)
        self.scale_spline = nn.Parameter(torch.ones(self.in_dim, self.out_dim))


class KAN(nn.Module):
    """Una KAN completa: pila de capas KAN, KAN(x) = (Phi_{L-1} o ... o Phi_0)(x)."""

    def __init__(self, widths, grid_size=3, k=3, grid_range=(0, 1)):
        super().__init__()
        self.widths = widths
        self.k = k
        self.grid_range = grid_range
        self.layers = nn.ModuleList([
            KANLayer(widths[i], widths[i + 1], grid_size=grid_size, k=k, grid_range=grid_range)
            for i in range(len(widths) - 1)
        ])

    def forward(self, x):
        phis = []
        for layer in self.layers:
            x, phi = layer(x)
            phis.append(phi)
        return x, phis

    def refine_grids(self, x_sample, new_grid_size):
        """Propaga muestras capa a capa y extiende la grid de cada una."""
        x = x_sample
        for layer in self.layers:
            layer.refine_grid(x, new_grid_size)
            x, _ = layer(x)


def sparsity_reg(phis, lamb_l1=1.0, lamb_entropy=1.0):
    """Regularizacion de dispersion (L1 + entropia) sobre las activaciones de cada capa, que
    empuja a la red hacia soluciones dispersas antes de la poda (parte del entrenamiento estandar de PyKAN)."""
    reg = 0.0
    for phi in phis:
        l1 = phi.abs().mean(dim=0)
        l1_norm = l1.sum()
        p = l1 / (l1_norm + 1e-8)
        entropy = -(p * torch.log(p + 1e-8)).sum()
        reg = reg + lamb_l1 * l1_norm + lamb_entropy * entropy
    return reg


def rmse(pred, target):
    return torch.sqrt(torch.mean((pred - target) ** 2))


def train_kan(model, X_tr, y_tr, X_te, y_te, steps=100, lr=1e-2, lamb=1e-3, verbose_every=50, label=''):
    """Entrenamiento con Adam (el paper/PyKAN usan LBFGS; Adam es mas robusto para una
    implementacion propia sin busqueda de linea, ver nota honesta al final)."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    hist = {'train': [], 'test': []}
    for step in range(steps):
        optimizer.zero_grad()
        pred, phis = model(X_tr)
        loss = torch.mean((pred - y_tr) ** 2) + lamb * sparsity_reg(phis)
        if not torch.isfinite(loss):
            raise RuntimeError('perdida no finita (NaN/Inf) durante el entrenamiento')
        loss.backward()
        optimizer.step()
        with torch.no_grad():
            tr = rmse(pred, y_tr).item()
            te = rmse(model(X_te)[0], y_te).item()
        hist['train'].append(tr)
        hist['test'].append(te)
        if step % verbose_every == 0 or step == steps - 1:
            print(f'  [{label}] paso {step:4d} | train RMSE={tr:.4e} | test RMSE={te:.4e}')
    return hist


# prueba rapida de forma
model_test = KAN([8, 4, 1], grid_size=3, k=3, grid_range=(0, 1))
x_test = torch.rand(5, 8)
y_test, phis_test = model_test(x_test)
print('KAN([8,4,1]) -> salida', y_test.shape, '| activaciones por capa:', [p.shape for p in phis_test])
print(f'{sum(p.numel() for p in model_test.parameters())} parametros entrenables')

## 2. Poda de aristas y nodos (Seccion 4.2): umbrales 0.03 / 0.01

Siguiendo la Seccion 4.2, calculamos para cada arista su puntuacion $|\phi_{i,j}|_1$ (magnitud media de la activacion sobre el conjunto de entrenamiento) y ponemos a cero (podamos) las aristas con puntuacion menor que **0.03**, el umbral por defecto de `PyKAN`. Para los nodos ocultos, calculamos su puntuacion de entrada y de salida (el maximo de las puntuaciones de sus aristas entrantes/salientes) y eliminamos por completo los nodos cuyas dos puntuaciones caen bajo **0.01**, reconstruyendo una KAN mas pequena. Esto solo se implementa para redes de exactamente 2 capas KAN (forma $[n_0,n_1,n_2]$), que es la forma de nuestras dos arquitecturas ($[8,4,1]$ para HTGR y $[6,1,1]$ para CHF).

In [ ]:
@torch.no_grad()
def edge_scores(model, x_sample):
    """Puntuacion |phi_{i,j}|_1 (magnitud media de la activacion) de cada arista de cada capa."""
    _, phis = model(x_sample)
    return [phi.abs().mean(dim=0) for phi in phis]   # lista de (in_dim, out_dim)


@torch.no_grad()
def apply_edge_pruning(model, x_sample, theta_edge=0.03):
    """Poda de aristas (Sec. 4.2): pone a cero la mascara de las aristas con puntuacion < theta_edge."""
    scores = edge_scores(model, x_sample)
    n_pruned = 0
    for layer, sc in zip(model.layers, scores):
        mask = (sc >= theta_edge).to(sc.dtype)
        n_pruned += int((layer.edge_mask * (1 - mask)).sum().item())
        layer.edge_mask = layer.edge_mask * mask
    return n_pruned


@torch.no_grad()
def apply_node_pruning(model, x_sample, theta_node=0.01):
    """Poda de nodos (Sec. 4.2): valido solo para una KAN de 2 capas [n0,n1,n2]. Elimina los
    nodos ocultos cuya puntuacion de entrada Y de salida caen ambas bajo theta_node."""
    assert len(model.widths) == 3, 'poda de nodos implementada solo para KAN de 2 capas [n0,n1,n2]'
    scores = edge_scores(model, x_sample)
    incoming = scores[0].amax(dim=0)   # (n1,) mejor arista de entrada a cada nodo oculto
    outgoing = scores[1].amax(dim=1)   # (n1,) mejor arista de salida desde cada nodo oculto
    keep = (incoming >= theta_node) | (outgoing >= theta_node)
    idx = keep.nonzero(as_tuple=True)[0]
    if len(idx) == 0:                                    # nunca dejar la red sin ningun nodo
        idx = torch.tensor([int(torch.argmax(incoming + outgoing))])
    if len(idx) == model.widths[1]:
        return model, idx                                 # nada que podar

    n0, _, n2 = model.widths
    n1_new = len(idx)
    grid_size = model.layers[0].grid.shape[1] - 2 * model.k - 1
    pruned = KAN([n0, n1_new, n2], grid_size=grid_size, k=model.k, grid_range=model.grid_range)
    l0, l1 = model.layers[0], model.layers[1]
    pruned.layers[0].coef.copy_(l0.coef[:, idx, :])
    pruned.layers[0].scale_base.copy_(l0.scale_base[:, idx])
    pruned.layers[0].scale_spline.copy_(l0.scale_spline[:, idx])
    pruned.layers[0].edge_mask.copy_(l0.edge_mask[:, idx])
    pruned.layers[0].grid = l0.grid.clone()

    pruned.layers[1].coef.copy_(l1.coef[idx, :, :])
    pruned.layers[1].scale_base.copy_(l1.scale_base[idx, :])
    pruned.layers[1].scale_spline.copy_(l1.scale_spline[idx, :])
    pruned.layers[1].edge_mask.copy_(l1.edge_mask[idx, :])
    pruned.layers[1].grid = l1.grid[idx].clone()
    return pruned, idx


def metrics(y_true, y_pred):
    """MAE, RMSE, R2 y MAPE(%) entre valores reales y predichos."""
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    mae = np.mean(np.abs(y_true - y_pred))
    rmse_ = np.sqrt(np.mean((y_true - y_pred) ** 2))
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2) + 1e-12
    r2 = 1 - ss_res / ss_tot
    mape = np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + 1e-12))) * 100
    return dict(MAE=mae, RMSE=rmse_, R2=r2, MAPE=mape)


def minmax_fit(X):
    return X.min(axis=0), X.max(axis=0)


def minmax_apply(X, lo, hi):
    return (X - lo) / (hi - lo + 1e-12)


def minmax_inv(Xs, lo, hi):
    return Xs * (hi - lo) + lo


print('funciones de poda y metricas definidas')

## 3. Identificacion simbolica: biblioteca de funciones candidatas

Tras la poda, el paper convierte cada arista superviviente en una funcion de la biblioteca simbolica de `PyKAN` (`SYMBOLIC_LIB`: `sin`, `cos`, `tan`, `exp`, `log`, `tanh`, `arctan`, polinomios, etc.), ajustando los parametros de afinidad de cada candidata por regresion. Implementamos una version reducida de ese mismo mecanismo: para cada arista no podada, probamos un conjunto de familias candidatas (elegidas para incluir exactamente las que aparecen en las Ec. 7 y 8 publicadas por el paper: `cos`, `tan`, `atanh`, gaussianas de la forma $\exp(-a(x-b)^2)$, ademas de `tanh`, cuadraticas y lineales) con `scipy.optimize.curve_fit`, y nos quedamos con la de mayor $R^2$. Con las familias identificadas, `symbolic_from_kan` ensambla la formula completa con `sympy`: para cada nodo oculto suma las funciones de sus aristas de entrada, y despues aplica la funcion de la arista de salida.

In [ ]:
def safe_tan(x, a, b, c, d):
    return c * np.tan(np.clip(a * x + b, -1.5, 1.5)) + d


def safe_atanh(x, a, b, c, d):
    return c * np.arctanh(np.clip(a * x + b, -0.999, 0.999)) + d


# (funcion, p0, limites (lo, hi) por parametro) - los limites evitan amplitudes/frecuencias
# desbocadas que ajustan bien en el rango de entrenamiento pero extrapolan mal en test.
CANDIDATES = {
    'lin': (lambda x, a, b: a * x + b, [1.0, 0.0], ([-50, -50], [50, 50])),
    'quad': (lambda x, a, b, c: a * x ** 2 + b * x + c, [1.0, 0.0, 0.0], ([-50, -50, -50], [50, 50, 50])),
    'cos': (lambda x, a, b, c, d: c * np.cos(a * x + b) + d, [3.0, 0.0, 1.0, 0.0],
            ([-30, -20, -10, -10], [30, 20, 10, 10])),
    'gauss': (lambda x, a, b, c, d: c * np.exp(-a * (x - b) ** 2) + d, [10.0, 0.5, 1.0, 0.0],
              ([1e-3, -3, -10, -10], [80, 3, 10, 10])),
    'tan': (safe_tan, [1.0, 0.0, 1.0, 0.0], ([-30, -20, -10, -10], [30, 20, 10, 10])),
    'atanh': (safe_atanh, [1.0, 0.0, 1.0, 0.0], ([-10, -10, -10, -10], [10, 10, 10, 10])),
    'tanh': (lambda x, a, b, c, d: c * np.tanh(a * x + b) + d, [1.0, 0.0, 1.0, 0.0],
             ([-20, -20, -10, -10], [20, 20, 10, 10])),
}


def fit_symbolic_edge(x, y, maxfev=4000):
    """Ajusta varias familias candidatas (con limites de parametro, ver CANDIDATES) y devuelve la
    de mayor R2 (analogo a suggest_symbolic de PyKAN)."""
    best = ('lin', -np.inf, [0.0, float(np.mean(y))])
    for name, (fn, p0, bounds) in CANDIDATES.items():
        try:
            popt, _ = curve_fit(fn, x, y, p0=p0, bounds=bounds, maxfev=maxfev)
            y_hat = fn(x, *popt)
            if not np.all(np.isfinite(y_hat)):
                continue
            ss_res = np.sum((y - y_hat) ** 2)
            ss_tot = np.sum((y - y.mean()) ** 2) + 1e-12
            r2 = 1 - ss_res / ss_tot
            if r2 > best[1]:
                best = (name, r2, popt)
        except Exception:
            continue
    return best


def make_sym_expr(name, popt, var):
    """Construye la expresion sympy de la familia identificada (parametros redondeados a 5 cifras)."""
    p = [round(float(v), 5) for v in popt]
    if name == 'lin':
        a, b = p
        return a * var + b
    if name == 'quad':
        a, b, c = p
        return a * var ** 2 + b * var + c
    if name == 'cos':
        a, b, c, d = p
        return c * sp.cos(a * var + b) + d
    if name == 'gauss':
        a, b, c, d = p
        return c * sp.exp(-a * (var - b) ** 2) + d
    if name == 'tan':
        a, b, c, d = p
        return c * sp.tan(a * var + b) + d
    if name == 'atanh':
        a, b, c, d = p
        return c * sp.atanh(a * var + b) + d
    if name == 'tanh':
        a, b, c, d = p
        return c * sp.tanh(a * var + b) + d
    return sp.nan


def symbolic_from_kan(model, x_sample, verbose=True):
    """Ajusta cada arista no podada de una KAN de 2 capas a una familia candidata y ensambla la
    formula simbolica completa (analogo a auto_symbolic de PyKAN, Sec. 4.2). Se ajusta sobre los
    valores EMPIRICOS reales (propagando x_sample capa a capa), no un barrido sintetico en un
    dominio asumido: el valor de un nodo oculto no esta acotado a [0,1], asi que barrer un rango
    asumido puede extrapolar de forma catastrofica."""
    n0, n1, n2 = model.widths
    syms = sp.symbols(f'x1:{n0 + 1}')
    x_np = x_sample.numpy()

    l0 = model.layers[0]
    with torch.no_grad():
        h, phi0 = l0(x_sample)          # h: (batch,n1) valores reales de los nodos ocultos
    hidden_exprs = [0 for _ in range(n1)]
    for j in range(n1):
        for i in range(n0):
            if l0.edge_mask[i, j].item() == 0:
                continue
            name, r2, popt = fit_symbolic_edge(x_np[:, i], phi0[:, i, j].numpy())
            if verbose:
                print(f'  capa0 arista (x{i + 1} -> nodo{j}): {name}, R2={r2:.4f}')
            hidden_exprs[j] = hidden_exprs[j] + make_sym_expr(name, popt, syms[i])

    l1 = model.layers[1]
    with torch.no_grad():
        _, phi1 = l1(h)
    h_np = h.numpy()
    final_expr = 0
    hid_sym = sp.symbols(f'h1:{n1 + 1}')
    for o in range(n2):
        for j in range(n1):
            if l1.edge_mask[j, o].item() == 0:
                continue
            name, r2, popt = fit_symbolic_edge(h_np[:, j], phi1[:, j, o].numpy())
            if verbose:
                print(f'  capa1 arista (nodo{j} -> y{o + 1}): {name}, R2={r2:.4f}')
            final_expr = final_expr + make_sym_expr(name, popt, hid_sym[j])

    subs_map = {hid_sym[j]: hidden_exprs[j] for j in range(n1)}
    # Nota: deliberadamente NO llamamos a sp.simplify() aqui. Con >1 nodo oculto y familias
    # transcendentales mixtas (tan/atanh/gaussianas anidadas), sympy puede tardar minutos u horas
    # intentando simplificar la expresion compuesta. sp.N()/lambdify() no necesitan la forma
    # simplificada para evaluar numericamente, asi que solo sustituimos.
    full_expr = final_expr.subs(subs_map)
    return full_expr, syms


print('funciones de identificacion simbolica definidas')

## 4. Caso 1 - HTGR: flujo neutronico del micro-reactor HOLOS-Quad

El primer caso de estudio del paper (Seccion 3) usa el benchmark **HTGR Micro-Core Quadrant Power** de `pyMAISE`: 751 simulaciones Serpent Monte Carlo del reactor HOLOS-Quad, donde 8 tambores de control ($\theta_1,\ldots,\theta_8$) se rotan y se mide el flujo neutronico normalizado resultante en los 4 cuadrantes del nucleo. El paper entrena con las 4 salidas pero, "para brevedad", solo reporta metricas sobre el primer cuadrante ($Q1$); seguimos la misma convencion aqui.

Descargamos **directamente el fichero real** `microreactor.csv` desde el repositorio Zenodo publico que aloja los datasets de `pyMAISE` (el mismo benchmark citado como referencia [6] del paper, DOI 10.5281/zenodo.20140559). Si la descarga falla por falta de conexion, usamos como respaldo datos sinteticos generados directamente a partir de la propia **Ec. 7** que el paper publica (su ecuacion simbolica final para $Q1$), anadiendo ruido gaussiano pequeno; en ese caso lo indicamos explicitamente en la salida.

**Simplificacion declarada:** el paper multiplica su dataset x4 explotando la simetria rotacional de los 4 cuadrantes (751 -> 3004 muestras). No reproducimos esa augmentacion (requeriria asumir una correspondencia exacta tambor-cuadrante que el paper no detalla); usamos directamente las 751 muestras reales, divididas 80/20 en entrenamiento/prueba.

In [ ]:
HTGR_URL = 'https://zenodo.org/records/20140559/files/microreactor.csv?download=1'
THETA_COLS = [f'theta{i}' for i in range(1, 9)]


def eq7_htgr(Xn):
    """Ec. (7) del paper: formula simbolica publicada para Q1, en el dominio normalizado [0,1]^8."""
    x1, x2, x3, x4, x5, x6, x7, x8 = [Xn[:, i] for i in range(8)]
    y = (-0.0006 * np.cos(6.7502 * x8 + 2.2243) + 0.2494
         - 0.0013 * np.exp(-12.96 * (0.5389 - x3) ** 2)
         - 0.0041 * np.exp(-12.96 * (0.5211 - x5) ** 2)
         + 0.0073 * np.exp(-13.4771 * (0.5074 - x2) ** 2)
         - 0.0023 * np.exp(-13.0714 * (0.5048 - x7) ** 2)
         - 0.0037 * np.exp(-13.8063 * (0.4958 - x6) ** 2)
         - 0.0023 * np.exp(-12.2786 * (0.4898 - x4) ** 2)
         + 0.0076 * np.exp(-12.5821 * (0.4692 - x1) ** 2))
    return y


def cargar_htgr(n_sint=751, seed=0):
    """Descarga microreactor.csv (Zenodo, pyMAISE). Si falla, genera un respaldo sintetico
    fiel a la Ec. 7 del paper, en un dominio fisico plausible (radianes, 0 a 2*pi por tambor)."""
    try:
        req = urllib.request.Request(HTGR_URL, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=20) as r:
            raw = r.read()
        df = pd.read_csv(io.BytesIO(raw), encoding='utf-8-sig')
        X = df[THETA_COLS].values.astype(np.float64)
        y = df['fluxQ1'].values.astype(np.float64)
        return X, y, 'real (Zenodo microreactor.csv, pyMAISE, 751 muestras Serpent MC)'
    except Exception as e:
        print('descarga HTGR fallida, usando respaldo sintetico basado en la Ec. 7:', repr(e))
        rng = np.random.default_rng(seed)
        Xn = rng.uniform(0, 1, size=(n_sint, 8))
        y_norm = eq7_htgr(Xn) + rng.normal(0, 0.0005, n_sint)
        X = Xn * 2 * np.pi                                       # dominio fisico ilustrativo (radianes)
        y_phys = 2.45e19 + (y_norm - y_norm.min()) / (y_norm.max() - y_norm.min() + 1e-12) * (2.73e19 - 2.45e19)
        return X, y_phys, 'sintetico (RESPALDO, derivado de la Ec. 7 del paper, sin descarga real)'


X_htgr, y_htgr, fuente_htgr = cargar_htgr()
print('Fuente de datos HTGR:', fuente_htgr)
print('N total =', len(y_htgr), '| rango fluxQ1 = [%.3e, %.3e]' % (y_htgr.min(), y_htgr.max()))

n = len(y_htgr)
idx_perm = np.random.default_rng(1).permutation(n)
n_test = int(0.2 * n)
te_idx_htgr, tr_idx_htgr = idx_perm[:n_test], idx_perm[n_test:]

lo_x_htgr, hi_x_htgr = minmax_fit(X_htgr[tr_idx_htgr])
lo_y_htgr, hi_y_htgr = y_htgr[tr_idx_htgr].min(), y_htgr[tr_idx_htgr].max()
Xn_tr_htgr = minmax_apply(X_htgr[tr_idx_htgr], lo_x_htgr, hi_x_htgr)
Xn_te_htgr = minmax_apply(X_htgr[te_idx_htgr], lo_x_htgr, hi_x_htgr)
yn_tr_htgr = (y_htgr[tr_idx_htgr] - lo_y_htgr) / (hi_y_htgr - lo_y_htgr)
yn_te_htgr = (y_htgr[te_idx_htgr] - lo_y_htgr) / (hi_y_htgr - lo_y_htgr)

Xtr_htgr = torch.tensor(Xn_tr_htgr)
Xte_htgr = torch.tensor(Xn_te_htgr)
ytr_htgr = torch.tensor(yn_tr_htgr).view(-1, 1)
yte_htgr = torch.tensor(yn_te_htgr).view(-1, 1)
print('train:', Xtr_htgr.shape, '| test:', Xte_htgr.shape)

## 5. Entrenamiento KAN $[8,4,1]$ para el flujo neutronico Q1

Reproducimos el protocolo exacto de la Seccion 4.2: KAN inicial con grid $G=3$ (100 epocas) $\to$ poda de aristas/nodos (umbrales 0.03/0.01) $\to$ 100 epocas mas $\to$ refinamiento de grid $G=5,10,25$ (100 epocas cada etapa), 500 epocas en total. Usamos la arquitectura $[8,4,1]$: el paper reporta como mejor configuracion `[8,4]` (8 entradas -> 4 salidas, ya que originalmente predice los 4 cuadrantes a la vez); como aqui solo modelamos $Q1$, adaptamos esa anchura oculta de 4 a una KAN de 2 capas $[8,4,1]$ (8 entradas -> 4 nodos ocultos -> 1 salida), manteniendo el ancho reportado como optimo.

In [ ]:
model_htgr = KAN([8, 4, 1], grid_size=3, k=3, grid_range=(0, 1))
print('=== Etapa 1: grid=3, 100 epocas ===')
train_kan(model_htgr, Xtr_htgr, ytr_htgr, Xte_htgr, yte_htgr,
          steps=100, lr=1e-2, lamb=1e-3, verbose_every=25, label='htgr-g3-init')

n_pruned_edges = apply_edge_pruning(model_htgr, Xtr_htgr, theta_edge=0.03)
model_htgr, kept_idx_htgr = apply_node_pruning(model_htgr, Xtr_htgr, theta_node=0.01)
print(f'\nPoda: {n_pruned_edges} aristas eliminadas | nodos ocultos conservados: '
      f'{kept_idx_htgr.tolist()} de 4\n')

print('=== Etapa 2: red podada, 100 epocas mas ===')
train_kan(model_htgr, Xtr_htgr, ytr_htgr, Xte_htgr, yte_htgr,
          steps=100, lr=1e-2, lamb=1e-4, verbose_every=25, label='htgr-post-poda')

historial_htgr = {'train': [], 'test': []}
for g in [5, 10, 25]:
    print(f'\n=== Refinamiento de grid: G={g}, 100 epocas ===')
    with torch.no_grad():
        model_htgr.refine_grids(Xtr_htgr, g)
    h = train_kan(model_htgr, Xtr_htgr, ytr_htgr, Xte_htgr, yte_htgr,
                   steps=100, lr=5e-3, lamb=1e-4, verbose_every=25, label=f'htgr-grid{g}')
    historial_htgr['train'] += h['train']
    historial_htgr['test'] += h['test']

plt.figure(figsize=(6, 4))
plt.semilogy(historial_htgr['train'], label='train RMSE')
plt.semilogy(historial_htgr['test'], label='test RMSE')
plt.xlabel('paso (dentro de la fase de refinamiento de grid)')
plt.ylabel('RMSE (normalizado)')
plt.title('KAN HTGR: refinamiento de grid G=5->10->25')
plt.legend()
plt.tight_layout()
plt.show()

## 6. Resultados HTGR: KAN vs FNN vs ecuacion simbolica del paper (Tabla I)

Entrenamos una FNN simple (2 capas ocultas, activacion `tanh`) con el mismo numero total de epocas, como el baseline mas directo frente a la KAN (el paper usa una FNN con arquitectura buscada por `pyMAISE`; aqui usamos una FNN generica de tamano comparable, ver nota honesta al final). Comparamos ambas contra los datos de test en el diagrama diagonal (predicho vs. real, analogo a la Fig. 3(a) del paper) y en metricas MAE/RMSE/R2/MAPE.

In [ ]:
class FNN(nn.Module):
    """FNN generica de 2 capas ocultas con tanh, usada como baseline directo frente a la KAN
    (el paper usa una FNN afinada con pyMAISE; esta es una version simple, ver nota honesta)."""
    def __init__(self, n_in, n_hidden, n_out=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, n_hidden), nn.Tanh(),
            nn.Linear(n_hidden, n_hidden), nn.Tanh(),
            nn.Linear(n_hidden, n_out))

    def forward(self, x):
        return self.net(x)


def entrenar_fnn(n_in, n_hidden, X_tr, y_tr, epochs=500, lr=5e-3, seed=0):
    torch.manual_seed(seed)
    fnn = FNN(n_in, n_hidden)
    opt = torch.optim.Adam(fnn.parameters(), lr=lr)
    for _ in range(epochs):
        opt.zero_grad()
        loss = torch.mean((fnn(X_tr) - y_tr) ** 2)
        loss.backward()
        opt.step()
    return fnn


fnn_htgr = entrenar_fnn(8, 8, Xtr_htgr, ytr_htgr, epochs=500)

with torch.no_grad():
    pred_kan_htgr = minmax_inv(model_htgr(Xte_htgr)[0].numpy().ravel(), lo_y_htgr, hi_y_htgr)
    pred_fnn_htgr = minmax_inv(fnn_htgr(Xte_htgr).numpy().ravel(), lo_y_htgr, hi_y_htgr)
y_te_htgr_phys = minmax_inv(yn_te_htgr, lo_y_htgr, hi_y_htgr)

m_kan_htgr = metrics(y_te_htgr_phys, pred_kan_htgr)
m_fnn_htgr = metrics(y_te_htgr_phys, pred_fnn_htgr)

tabla_htgr = pd.DataFrame([
    dict(modelo='KAN [8,4,1] (este cuaderno)', **m_kan_htgr),
    dict(modelo='FNN [8,8,8,1] (este cuaderno)', **m_fnn_htgr),
]).set_index('modelo')
print('Metricas HTGR (unidades fisicas, fluxQ1):')
print(tabla_htgr.round(4))

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(y_te_htgr_phys, pred_kan_htgr, s=18, alpha=0.6, label=f"KAN (R2={m_kan_htgr['R2']:.3f})")
ax.scatter(y_te_htgr_phys, pred_fnn_htgr, s=18, alpha=0.6, label=f"FNN (R2={m_fnn_htgr['R2']:.3f})")
lims = [y_te_htgr_phys.min(), y_te_htgr_phys.max()]
ax.plot(lims, lims, 'k--', lw=1, label='y = x')
ax.set_xlabel('fluxQ1 real')
ax.set_ylabel('fluxQ1 predicho')
ax.set_title('HTGR: validacion diagonal en test (cf. Fig. 3(a) del paper)')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Identificacion simbolica del KAN HTGR y comparacion con la Ec. 7 del paper

Convertimos la KAN entrenada en una formula cerrada (Seccion 3) y evaluamos su fidelidad con el $R^2$ frente a las predicciones de la propia red en el conjunto de test (analogo a la columna "Symbolic Equation" de la Tabla I, que el paper reporta con $R^2=0.9643$ para HTGR). Imprimimos tambien la Ec. 7 original del paper como referencia directa: si nuestra formula recupera una estructura similar (suma de terminos gaussianos/coseno, uno por cada tambor), es evidencia de que el mecanismo de interpretabilidad central del paper -convertir pesos de spline en una ecuacion legible- se reproduce correctamente, aunque los coeficientes numericos exactos no coincidan (dependen de la semilla, los datos y el presupuesto de entrenamiento).

In [ ]:
expr_htgr, syms_htgr = symbolic_from_kan(model_htgr, Xtr_htgr)
print('\nFormula simbolica HTGR reconstruida (coeficientes redondeados a 4 cifras):')
sp.pprint(sp.N(expr_htgr, 4))

f_htgr = sp.lambdify(syms_htgr, expr_htgr, 'numpy')
y_sym_te_htgr = np.asarray(f_htgr(*[Xn_te_htgr[:, i] for i in range(8)]), dtype=np.float64)
if y_sym_te_htgr.ndim == 0:
    y_sym_te_htgr = np.full(len(yn_te_htgr), float(y_sym_te_htgr))
r2_sym_vs_datos = metrics(yn_te_htgr, y_sym_te_htgr)['R2']
r2_sym_vs_kan = metrics(model_htgr(Xte_htgr)[0].detach().numpy().ravel(), y_sym_te_htgr)['R2']
print(f'\nR2 formula simbolica vs. datos reales de test: {r2_sym_vs_datos:.4f}')
print(f'R2 formula simbolica vs. predicciones del KAN en test: {r2_sym_vs_kan:.4f}')
print('(el paper reporta R2=0.9643 para su ecuacion simbolica de HTGR frente a los datos reales, Tabla I)')

print('\nEc. 7 del paper (referencia, dominio normalizado x1..x8 en [0,1]):')
print('Y_HTGR = -0.0006*cos(6.7502*x8+2.2243) + 0.2494')
print('         - 0.0013*exp(-12.96*(0.5389-x3)^2) - 0.0041*exp(-12.96*(0.5211-x5)^2)')
print('         + 0.0073*exp(-13.4771*(0.5074-x2)^2) - 0.0023*exp(-13.0714*(0.5048-x7)^2)')
print('         - 0.0037*exp(-13.8063*(0.4958-x6)^2) - 0.0023*exp(-12.2786*(0.4898-x4)^2)')
print('         + 0.0076*exp(-12.5821*(0.4692-x1)^2)')

## 8. Caso 2 - CHF: flujo critico de calor

El segundo caso de estudio del paper (Seccion 3) predice el **flujo critico de calor (CHF)** a partir de 6 variables termohidraulicas: diametro del canal $D$ (m), longitud calentada $L$ (m), presion $P$ (kPa), flujo masico $G$ (kg/m²s), temperatura de entrada $T_{in}$ (°C) y calidad de equilibrio $X_e$ (-). El dataset original (tablas de referencia de la NRC/Groeneveld 2006, citadas en las referencias [8]-[9] del paper) **no es publico**; el propio repositorio de los autores lo confirma explicitamente: *"Since the real CHF dataset used in this analysis is not public, we have provided synthetic versions of this dataset"* (README de `aims-umich/2025-panczyk-kan`).

Descargamos por tanto la **version sintetica oficial** que el equipo de `pyMAISE` publica en el mismo Zenodo (`chf_train_synth.csv`, 2000 muestras; `chf_test_synth.csv`, 500 muestras) — el mismo sustituto que usan los propios autores cuando no se dispone del dataset real. Si la descarga falla, usamos como respaldo datos sinteticos generados directamente a partir de la **Ec. 8** publicada por el paper (su ecuacion simbolica final para el CHF), en cuyo caso lo indicamos explicitamente.

In [ ]:
CHF_TRAIN_URL = 'https://zenodo.org/records/20140559/files/chf_train_synth.csv?download=1'
CHF_TEST_URL = 'https://zenodo.org/records/20140559/files/chf_test_synth.csv?download=1'
CHF_COLS = ['D (m)', 'L (m)', 'P (kPa)', 'G (kg m-2s-1)', 'Tin (C)', 'Xe (-)']


def eq8_chf(Xn):
    """Ec. (8) del paper: formula simbolica publicada para CHF, en el dominio normalizado [0,1]^6."""
    x1, x2, x3, x4, x5, x6 = [Xn[:, i] for i in range(6)]
    h = (0.6936 * np.cos(1.5813 * x1 - 4.589)
         - 0.0589 * np.tan(2.7642 * x3 - 4.5831)
         - 0.3475 * np.tan(2.212 * x4 - 1.4029)
         + 0.6529 * np.tan(0.6 * x5 + 9.9968)
         + 0.9714 * np.arctanh(np.clip(1.0 * x2 - 0.992, -0.999, 0.999))
         + 1.3965
         + 0.4367 * np.exp(-6.1413 * (0.2107 - x6) ** 2))
    return 0.6975 - 0.6996 * np.tanh(h)


def cargar_chf(n_sint=2500, seed=0):
    """Descarga chf_train/test_synth.csv (Zenodo, pyMAISE, version sintetica OFICIAL usada por
    los propios autores). Si falla, genera un respaldo sintetico propio fiel a la Ec. 8."""
    try:
        dfs = []
        for url in (CHF_TRAIN_URL, CHF_TEST_URL):
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=20) as r:
                raw = r.read()
            dfs.append(pd.read_csv(io.BytesIO(raw), encoding='utf-8-sig'))
        df_train, df_test = dfs
        Xtr = df_train[CHF_COLS].values.astype(np.float64)
        ytr = df_train['CHF (kW m-2)'].values.astype(np.float64)
        Xte = df_test[CHF_COLS].values.astype(np.float64)
        yte = df_test['CHF (kW m-2)'].values.astype(np.float64)
        return Xtr, ytr, Xte, yte, 'real (Zenodo chf_train/test_synth.csv, sintetico OFICIAL de pyMAISE)'
    except Exception as e:
        print('descarga CHF fallida, usando respaldo sintetico propio basado en la Ec. 8:', repr(e))
        rng = np.random.default_rng(seed)
        Xn = rng.uniform(0.02, 0.98, size=(n_sint, 6))
        y_norm = eq8_chf(Xn) + rng.normal(0, 0.01, n_sint)
        y_phys = 130 + np.clip(y_norm, 0, 1) * (13345 - 130)
        rangos = dict(D=(0.005, 0.016), L=(0.3, 6.0), P=(100, 20000), G=(200, 8000), Tin=(20, 340), Xe=(-0.5, 0.7))
        cols = [lo + Xn[:, i] * (hi - lo) for i, (lo, hi) in enumerate(rangos.values())]
        X = np.stack(cols, axis=1)
        n_test = int(0.2 * n_sint)
        return (X[n_test:], y_phys[n_test:], X[:n_test], y_phys[:n_test],
                'sintetico (RESPALDO propio, derivado de la Ec. 8 del paper, sin descarga real)')


Xtr_raw_chf, ytr_raw_chf, Xte_raw_chf, yte_raw_chf, fuente_chf = cargar_chf()
print('Fuente de datos CHF:', fuente_chf)
print('N_train =', len(ytr_raw_chf), '| N_test =', len(yte_raw_chf))
print('rango CHF (kW/m2) = [%.1f, %.1f]' % (ytr_raw_chf.min(), ytr_raw_chf.max()))

lo_x_chf, hi_x_chf = minmax_fit(Xtr_raw_chf)
lo_y_chf, hi_y_chf = ytr_raw_chf.min(), ytr_raw_chf.max()
Xn_tr_chf = minmax_apply(Xtr_raw_chf, lo_x_chf, hi_x_chf)
Xn_te_chf = minmax_apply(Xte_raw_chf, lo_x_chf, hi_x_chf)
yn_tr_chf = (ytr_raw_chf - lo_y_chf) / (hi_y_chf - lo_y_chf)
yn_te_chf = (yte_raw_chf - lo_y_chf) / (hi_y_chf - lo_y_chf)

Xtr_chf = torch.tensor(Xn_tr_chf)
Xte_chf = torch.tensor(Xn_te_chf)
ytr_chf = torch.tensor(yn_tr_chf).view(-1, 1)
yte_chf = torch.tensor(yn_te_chf).view(-1, 1)
print('train:', Xtr_chf.shape, '| test:', Xte_chf.shape)

## 9. Entrenamiento KAN $[6,1,1]$ para el CHF (mismo protocolo)

El paper reporta la arquitectura `[6,1,1]` (6 entradas -> 1 nodo oculto -> 1 salida) como la de mejores metricas de test para CHF, con 25 intervalos de grid finales; coincide exactamente con la estructura que se deduce de la propia Ec. 8: una unica combinacion aditiva de 6 funciones univariadas (una por variable) seguida de una unica funcion de salida (una $\tanh$ afin). Aplicamos el mismo protocolo de entrenamiento-poda-refinamiento de grid que en HTGR.

In [ ]:
model_chf = KAN([6, 1, 1], grid_size=3, k=3, grid_range=(0, 1))
print('=== Etapa 1: grid=3, 100 epocas ===')
train_kan(model_chf, Xtr_chf, ytr_chf, Xte_chf, yte_chf,
          steps=100, lr=1e-2, lamb=1e-3, verbose_every=25, label='chf-g3-init')

n_pruned_edges_chf = apply_edge_pruning(model_chf, Xtr_chf, theta_edge=0.03)
model_chf, kept_idx_chf = apply_node_pruning(model_chf, Xtr_chf, theta_node=0.01)
print(f'\nPoda: {n_pruned_edges_chf} aristas eliminadas | nodos ocultos conservados: '
      f'{kept_idx_chf.tolist()} de 1\n')

print('=== Etapa 2: red podada, 100 epocas mas ===')
train_kan(model_chf, Xtr_chf, ytr_chf, Xte_chf, yte_chf,
          steps=100, lr=1e-2, lamb=1e-4, verbose_every=25, label='chf-post-poda')

historial_chf = {'train': [], 'test': []}
for g in [5, 10, 25]:
    print(f'\n=== Refinamiento de grid: G={g}, 100 epocas ===')
    with torch.no_grad():
        model_chf.refine_grids(Xtr_chf, g)
    h = train_kan(model_chf, Xtr_chf, ytr_chf, Xte_chf, yte_chf,
                   steps=100, lr=5e-3, lamb=1e-4, verbose_every=25, label=f'chf-grid{g}')
    historial_chf['train'] += h['train']
    historial_chf['test'] += h['test']

plt.figure(figsize=(6, 4))
plt.semilogy(historial_chf['train'], label='train RMSE')
plt.semilogy(historial_chf['test'], label='test RMSE')
plt.xlabel('paso (dentro de la fase de refinamiento de grid)')
plt.ylabel('RMSE (normalizado)')
plt.title('KAN CHF: refinamiento de grid G=5->10->25')
plt.legend()
plt.tight_layout()
plt.show()

## 10. Resultados CHF e identificacion simbolica (comparacion con la Ec. 8 y la Tabla I)

In [ ]:
fnn_chf = entrenar_fnn(6, 6, Xtr_chf, ytr_chf, epochs=500)

with torch.no_grad():
    pred_kan_chf = minmax_inv(model_chf(Xte_chf)[0].numpy().ravel(), lo_y_chf, hi_y_chf)
    pred_fnn_chf = minmax_inv(fnn_chf(Xte_chf).numpy().ravel(), lo_y_chf, hi_y_chf)
y_te_chf_phys = minmax_inv(yn_te_chf, lo_y_chf, hi_y_chf)

m_kan_chf = metrics(y_te_chf_phys, pred_kan_chf)
m_fnn_chf = metrics(y_te_chf_phys, pred_fnn_chf)

tabla_chf = pd.DataFrame([
    dict(modelo='KAN [6,1,1] (este cuaderno)', **m_kan_chf),
    dict(modelo='FNN [6,6,6,1] (este cuaderno)', **m_fnn_chf),
]).set_index('modelo')
print('Metricas CHF (unidades fisicas, kW/m2):')
print(tabla_chf.round(4))

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(y_te_chf_phys, pred_kan_chf, s=14, alpha=0.5, label=f"KAN (R2={m_kan_chf['R2']:.3f})")
ax.scatter(y_te_chf_phys, pred_fnn_chf, s=14, alpha=0.5, label=f"FNN (R2={m_fnn_chf['R2']:.3f})")
lims = [y_te_chf_phys.min(), y_te_chf_phys.max()]
ax.plot(lims, lims, 'k--', lw=1, label='y = x')
ax.set_xlabel('CHF real (kW/m2)')
ax.set_ylabel('CHF predicho (kW/m2)')
ax.set_title('CHF: validacion diagonal en test (cf. Fig. 3(b) del paper)')
ax.legend()
plt.tight_layout()
plt.show()

expr_chf, syms_chf = symbolic_from_kan(model_chf, Xtr_chf)
print('\nFormula simbolica CHF reconstruida (coeficientes redondeados a 4 cifras):')
sp.pprint(sp.N(expr_chf, 4))

f_chf = sp.lambdify(syms_chf, expr_chf, 'numpy')
y_sym_te_chf = np.asarray(f_chf(*[Xn_te_chf[:, i] for i in range(6)]), dtype=np.float64)
if y_sym_te_chf.ndim == 0:
    y_sym_te_chf = np.full(len(yn_te_chf), float(y_sym_te_chf))
r2_sym_vs_datos_chf = metrics(yn_te_chf, y_sym_te_chf)['R2']
print(f'\nR2 formula simbolica vs. datos reales de test: {r2_sym_vs_datos_chf:.4f}')
print('(el paper reporta R2=0.9757 para su ecuacion simbolica de CHF frente a los datos reales, Tabla I)')

print('\nEc. 8 del paper (referencia, dominio normalizado x1..x6 en [0,1]):')
print('Y_CHF = 0.6975 - 0.6996*tanh( 0.6936*cos(1.5813*x1-4.589)')
print('                              - 0.0589*tan(2.7642*x3-4.5831) - 0.3475*tan(2.212*x4-1.4029)')
print('                              + 0.6529*tan(0.6*x5+9.9968) + 0.9714*atanh(x2-0.992) + 1.3965')
print('                              + 0.4367*exp(-6.1413*(0.2107-x6)^2) )')

## 11. Explicabilidad: importancia de variables tipo SHAP (Seccion 4.3)

El paper aplica **Kernel SHAP** sobre la KAN y **DeepLIFT** sobre la FNN para comparar que variables considera importante cada modelo (Fig. 4-5), encontrando que la KAN se alinea mejor con la fisica esperada del problema. Implementar Kernel SHAP/DeepLIFT completos requiere la libreria `shap` y esta fuera del alcance de este cuaderno; en su lugar usamos un **proxy simple de importancia por permutacion**: para cada variable, barajamos sus valores en el conjunto de test y medimos cuanto aumenta el error del modelo (MAE). Una variable importante produce un aumento grande al ser barajada. No es Kernel SHAP ni DeepLIFT (no reparte contribuciones por Shapley ni por retropropagacion de diferencias), pero captura la misma idea cualitativa -que variables mueve mas la prediccion de cada modelo- y nos permite comparar KAN vs FNN en ambos casos de estudio.

Para HTGR, el paper predice que los tambores 1 y 2 (mas cercanos al cuadrante Q1) deberian ser mas importantes; para CHF, se espera que $L$, $G$ y $T_{in}$ dominen (relacionados directamente con la ebullicion y el flujo termico).

In [ ]:
def importancia_permutacion(predict_fn, X, y_true, n_repeats=5, seed=0):
    """Aumento del MAE al barajar cada columna de X (proxy simple de importancia, no es SHAP/DeepLIFT)."""
    rng = np.random.default_rng(seed)
    base_mae = np.mean(np.abs(predict_fn(X) - y_true))
    n_features = X.shape[1]
    importancias = np.zeros(n_features)
    for i in range(n_features):
        deltas = []
        for _ in range(n_repeats):
            X_perm = X.copy()
            X_perm[:, i] = rng.permutation(X_perm[:, i])
            mae_perm = np.mean(np.abs(predict_fn(X_perm) - y_true))
            deltas.append(mae_perm - base_mae)
        importancias[i] = np.mean(deltas)
    return importancias


def graficar_importancias(ax, nombres, imp_kan, imp_fnn, titulo):
    x = np.arange(len(nombres))
    ax.bar(x - 0.2, imp_kan, width=0.4, label='KAN (proxy)')
    ax.bar(x + 0.2, imp_fnn, width=0.4, label='FNN (proxy)')
    ax.set_xticks(x)
    ax.set_xticklabels(nombres, rotation=45, ha='right')
    ax.set_ylabel('aumento de MAE al permutar')
    ax.set_title(titulo)
    ax.legend()


# --- HTGR: predict_fn en unidades normalizadas (mismo dominio en el que se entreno) ---
def pred_kan_htgr_fn(X):
    with torch.no_grad():
        return model_htgr(torch.tensor(X))[0].numpy().ravel()


def pred_fnn_htgr_fn(X):
    with torch.no_grad():
        return fnn_htgr(torch.tensor(X)).numpy().ravel()


imp_kan_htgr = importancia_permutacion(pred_kan_htgr_fn, Xn_te_htgr.copy(), yn_te_htgr)
imp_fnn_htgr = importancia_permutacion(pred_fnn_htgr_fn, Xn_te_htgr.copy(), yn_te_htgr)
nombres_htgr = [f'theta{i}' for i in range(1, 9)]

# --- CHF: idem ---
def pred_kan_chf_fn(X):
    with torch.no_grad():
        return model_chf(torch.tensor(X))[0].numpy().ravel()


def pred_fnn_chf_fn(X):
    with torch.no_grad():
        return fnn_chf(torch.tensor(X)).numpy().ravel()


imp_kan_chf = importancia_permutacion(pred_kan_chf_fn, Xn_te_chf.copy(), yn_te_chf)
imp_fnn_chf = importancia_permutacion(pred_fnn_chf_fn, Xn_te_chf.copy(), yn_te_chf)
nombres_chf = ['D', 'L', 'P', 'G', 'Tin', 'Xe']

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
graficar_importancias(axes[0], nombres_htgr, imp_kan_htgr, imp_fnn_htgr,
                       'HTGR: importancia por permutacion (esperado: theta1, theta2 altas)')
graficar_importancias(axes[1], nombres_chf, imp_kan_chf, imp_fnn_chf,
                       'CHF: importancia por permutacion (esperado: L, G, Tin altas)')
plt.tight_layout()
plt.show()

print('Ranking HTGR (KAN):', [nombres_htgr[i] for i in np.argsort(-imp_kan_htgr)])
print('Ranking CHF (KAN):', [nombres_chf[i] for i in np.argsort(-imp_kan_chf)])

## 12. Comparacion final con el paper

| Aspecto | Paper (Tabla I, Ec. 7-8) | Este cuaderno |
|---|---|---|
| Libreria KAN | `PyKAN` 0.2.x (caja de herramientas completa) | Reimplementada en PyTorch (splines B, poda, refinamiento de grid, simbolico) |
| Arquitectura HTGR | `[8,4]` (multi-salida, 4 cuadrantes) | `[8,4,1]` (solo Q1) |
| Arquitectura CHF | `[6,1,1]` | `[6,1,1]` (identica) |
| Protocolo de entrenamiento | 100+100+100+100+100 epocas (grid 3->poda->3->5->10->25) | Igual |
| Umbrales de poda | aristas 0.03 / nodos 0.01 | Igual |
| Optimizador | LBFGS (por defecto de `PyKAN`) | Adam (mas robusto para una implementacion propia sin busqueda de linea) |
| Datos HTGR | 751 (x4 por simetria = 3004), Serpent MC | 751 reales (sin augmentacion), mismo origen (Zenodo/pyMAISE) |
| Datos CHF | NRC/Groeneveld 2006 real (no publico) | Version sintetica oficial de pyMAISE (2000+500), la misma que usan los propios autores cuando no hay datos reales |
| Explicabilidad | Kernel SHAP (KAN) / DeepLIFT (FNN) | Proxy de importancia por permutacion (ambos modelos) |
| Simbolico HTGR | $R^2=0.9643$ (Tabla I) | Ver Seccion 7 (variable, presupuesto reducido) |
| Simbolico CHF | $R^2=0.9757$ (Tabla I) | Ver Seccion 10 (variable, presupuesto reducido) |

### Nota honesta sobre los resultados

Este cuaderno reproduce fielmente el mecanismo central del paper -la capa KAN con splines B (Ec. 4-6), el protocolo completo de entrenamiento-poda-refinamiento de grid con los umbrales exactos (0.03/0.01, Sec. 4.2), y la conversion a formula simbolica- para los dos casos de estudio reales del paper (HTGR y CHF), pero **no es una replica numerica exacta**, por las siguientes razones, declaradas explicitamente:

- **Libreria.** No usamos `PyKAN` como caja negra (aunque esta disponible localmente y via `pip install pykan`): reimplementamos su mecanismo en PyTorch puro, siguiendo la misma convencion pedagogica que el resto de esta coleccion (`KAN Symbolic Regression.ipynb`, `MultKAN Conserved Quantities.ipynb`). Esto significa que no contamos con las optimizaciones internas de `PyKAN` (LBFGS con busqueda de linea, `suggest_symbolic`/`auto_symbolic` con biblioteca completa de funciones y ajuste de afinidad optimizado) y usamos, en su lugar, Adam y una busqueda reducida de 7 familias candidatas con `scipy.optimize.curve_fit`.
- **Datos HTGR.** Descargamos el fichero real `microreactor.csv` (751 muestras, el mismo benchmark citado por el paper); si la conexion a Zenodo falla en el momento de ejecutar este cuaderno, se usa automaticamente un respaldo sintetico derivado de la propia Ec. 7 del paper (verifique el mensaje `"Fuente de datos HTGR:"` en la salida de la Seccion 4 para saber cual se uso). No aplicamos la augmentacion x4 por simetria rotacional que usa el paper (751->3004 muestras), por lo que entrenamos con menos datos que el articulo original.
- **Datos CHF.** El dataset real (NRC/Groeneveld 2006) no es publico -lo confirma el propio repositorio de los autores-. Usamos la version sintetica **oficial** que el equipo de `pyMAISE` distribuye para este proposito exacto (el mismo sustituto que usan los propios autores del paper cuando no disponen del dataset real), con respaldo a una version sintetica propia basada en la Ec. 8 si la descarga falla.
- **Arquitectura HTGR adaptada.** El paper entrena una KAN multi-salida `[8,4]` que predice los 4 cuadrantes simultaneamente y solo reporta metricas de Q1; aqui, como modelamos unicamente Q1, usamos `[8,4,1]` (mismo ancho oculto reportado como optimo, una sola salida).
- **Identificacion simbolica.** Nuestro procedimiento ajusta cada arista superviviente a la mejor de 7 familias candidatas usando los valores **empiricos reales** de cada activacion (no un barrido en un dominio asumido, que en una version inicial de este cuaderno produjo formulas que extrapolaban de forma catastrofica para la capa de salida). Aun asi, al ser un ajuste marginal arista por arista (no la optimizacion conjunta y la biblioteca completa de `PyKAN`), el $R^2$ de la formula simbolica resultante frente a los datos reales de test es tipicamente mas bajo y mas variable (dependiente de la semilla) que el $R^2=0.9643$ (HTGR) / $0.9757$ (CHF) que reporta el paper en su Tabla I; consulte los valores impresos en las Secciones 7 y 10 de este cuaderno para el resultado concreto de esta ejecucion.
- **Explicabilidad.** Sustituimos Kernel SHAP (sobre la KAN) y DeepLIFT (sobre la FNN) -que requieren la libreria `shap` y serian costosos de implementar desde cero con garantias- por un proxy de importancia por permutacion, mas simple pero conceptualmente relacionado (mide cuanto se degrada la prediccion al romper la relacion entre una variable y la salida).
- **FNN baseline.** El paper usa una FNN cuya arquitectura fue buscada automaticamente por `pyMAISE` (hyperparameter tuning); aqui usamos una FNN generica de tamano comparable sin busqueda de hiperparametros, por lo que la comparacion KAN-vs-FNN de este cuaderno es indicativa, no una replica exacta de la Tabla I.

En conjunto, el mecanismo que el paper propone como su contribucion central -entrenar una KAN, podarla, y convertirla en una ecuacion simbolica interpretable que aproxima razonablemente bien los datos reales- se reproduce y se observa en ambos casos de estudio, con datos reales o oficiales cuando estan disponibles.